# InternVL Auditor v2 Cleaning Notebook

This notebook streams the dataset, audits all images with the auditor model, selects the top-10k most confident images per category, builds safety masks from the adversarial heatmap, and uploads the cleaned subset to the target Hugging Face repo.

## 0) Install Dependencies (Run Once)
If your environment already has these, you can skip this cell.

In [ ]:
# If needed, uncomment the next line.
# %pip install -U torch torchvision datasets huggingface_hub pillow opencv-python tqdm

## 1) Imports and Configuration

In [ ]:
import os
import io
import json
import math
import heapq
from typing import Dict, Any, Iterable, Tuple, List

import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset, Features, Value, ClassLabel, Image as HFImage, Sequence
from huggingface_hub import login
import getpass

from auditor_inference import AdversarialAuditor, CLASS_NAMES

In [ ]:
DATASET_ID = "ShreyashDhoot/internvl-auditor-v2"
REPO_ID = "ShreyashDhoot/inpainter-cleaning"

OUTPUT_DIR = "outputs_cleaned"
SCORES_PATH = os.path.join(OUTPUT_DIR, "scores.jsonl")
TOPK_PATH = os.path.join(OUTPUT_DIR, "topk_indices.json")

TOP_K = 10_000
HEATMAP_THRESHOLD = 0.6
DILATE_KERNEL = 5
DILATE_MAX_ITERS = 10

# If you want to test on fewer items, set a number here; use None for full stream.
MAX_ITEMS = None

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2) Initialize Auditor

In [ ]:
auditor = AdversarialAuditor(
    model_path=os.path.join(OUTPUT_DIR, "complete_auditor_best.pth"),
    vocab_path=os.path.join(OUTPUT_DIR, "vocab.json"),
)

auditor.model.eval()
device = next(auditor.model.parameters()).device

## 3) Streaming Helpers
This reads images and prompts from the dataset without downloading it.

In [ ]:
def extract_image_and_prompt(sample: Dict[str, Any]) -> Tuple[Image.Image, str]:
    # Common image keys in HF datasets
    image = None
    for key in ["image", "img", "pixel_values", "jpg", "png"]:
        if key in sample:
            image = sample[key]
            break
    if image is None:
        raise KeyError("No image field found in sample")

    # HF streaming often yields PIL.Image directly; handle dict-encoded images if needed
    if isinstance(image, dict) and "bytes" in image:
        image = Image.open(io.BytesIO(image["bytes"]))
    if not isinstance(image, Image.Image):
        image = Image.fromarray(np.array(image))

    prompt = None
    for key in ["prompt", "text", "caption", "instruction"]:
        if key in sample:
            prompt = sample[key]
            break
    if prompt is None:
        prompt = ""
    return image.convert("RGB"), str(prompt)


def audit_with_probs(image: Image.Image, prompt: str) -> Dict[str, Any]:
    img_tensor = auditor.transform(image).unsqueeze(0).to(device)
    tokens = auditor.tokenizer.encode(prompt).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = auditor.model(img_tensor, text_tokens=tokens, timestep=torch.zeros(1, 1).to(device))
    class_probs = F.softmax(outputs["class_logits"], dim=1)[0].cpu().numpy()
    pred_idx = int(np.argmax(class_probs))
    pred_label = CLASS_NAMES[pred_idx]
    pred_conf = float(class_probs[pred_idx])
    heatmap = outputs["adversarial_map"][0, 0].cpu().numpy()
    return {
        "prediction": pred_label,
        "prediction_idx": pred_idx,
        "prediction_confidence": pred_conf,
        "class_probs": class_probs,
        "heatmap": heatmap,
    }


def update_topk(heap: List[Tuple[float, int]], score: float, index: int, k: int):
    # Min-heap of (score, index)
    if len(heap) < k:
        heapq.heappush(heap, (score, index))
    else:
        if score > heap[0][0]:
            heapq.heapreplace(heap, (score, index))


def heatmap_to_binary(heatmap: np.ndarray, threshold: float, image_size: Tuple[int, int]) -> np.ndarray:
    h, w = image_size[1], image_size[0]
    resized = cv2.resize(heatmap, (w, h), interpolation=cv2.INTER_LINEAR)
    binary = (resized >= threshold).astype(np.uint8) * 255
    return binary


def apply_mask(image: Image.Image, binary_mask: np.ndarray) -> Image.Image:
    img = np.array(image)
    mask = (binary_mask > 0)
    img[mask] = 0
    return Image.fromarray(img)


def ensure_safe_mask(image: Image.Image, prompt: str, binary_mask: np.ndarray) -> np.ndarray:
    kernel = np.ones((DILATE_KERNEL, DILATE_KERNEL), np.uint8)
    current = binary_mask.copy()
    for _ in range(DILATE_MAX_ITERS + 1):
        masked = apply_mask(image, current)
        res = auditor.audit(masked, prompt)
        if res["prediction"] == "Safe":
            return current
        current = cv2.dilate(current, kernel, iterations=1)
    return current

## 4) Score All Images (Streaming) and Track Top-K Per Category

In [ ]:
with open(TOPK_PATH, "r", encoding="utf-8") as f:
    topk_indices = json.load(f)

selected_indices = set()

for name, items in topk_indices.items():
    selected_indices.update([idx for _, idx in items])

print("Selected indices:", len(selected_indices))

image_dir = os.path.join(OUTPUT_DIR, "images")
mask_dir = os.path.join(OUTPUT_DIR, "masks")
masked_dir = os.path.join(OUTPUT_DIR, "masked")
os.makedirs(image_dir, exist_ok=True)
os.makedirs(mask_dir, exist_ok=True)
os.makedirs(masked_dir, exist_ok=True)

rows = []
ds = load_dataset(DATASET_ID, split="train", streaming=True)

for idx, sample in tqdm(enumerate(ds), total=len(selected_indices)):
    if idx not in selected_indices:
        continue
    image, prompt = extract_image_and_prompt(sample)
    res = audit_with_probs(image, prompt)
    heatmap = res["heatmap"]
    binary = heatmap_to_binary(heatmap, HEATMAP_THRESHOLD, image.size)
    safe_mask = ensure_safe_mask(image, prompt, binary)
    masked_image = apply_mask(image, safe_mask)

    image_path = os.path.join(image_dir, f"{idx}.png")
    mask_path = os.path.join(mask_dir, f"{idx}.png")
    masked_path = os.path.join(masked_dir, f"{idx}.png")

    image.save(image_path)
    Image.fromarray(safe_mask).save(mask_path)
    masked_image.save(masked_path)

    one_hot = [1 if i == res["prediction_idx"] else 0 for i in range(len(CLASS_NAMES))]

    rows.append({
        "index": idx,
        "prompt": prompt,
        "category": res["prediction"],
        "category_confidence": float(res["prediction_confidence"]),
        "category_one_hot": one_hot,
        "original_image": image_path,
        "mask": mask_path,
        "masked_image": masked_path,
    })
    if len(rows) >= 30_000:
        break

print("Prepared rows:", len(rows))

## 5) Build Masks for the Selected 30,000 Images
We stream the dataset again and only process the selected indices.

In [ ]:
with open(TOPK_PATH, "r", encoding="utf-8") as f:
    topk_indices = json.load(f)

selected_indices = set()
for name, items in topk_indices.items():
    selected_indices.update([idx for _, idx in items])

print("Selected indices:", len(selected_indices))

image_dir = os.path.join(OUTPUT_DIR, "images")
mask_dir = os.path.join(OUTPUT_DIR, "masks")
masked_dir = os.path.join(OUTPUT_DIR, "masked")
os.makedirs(image_dir, exist_ok=True)
os.makedirs(mask_dir, exist_ok=True)
os.makedirs(masked_dir, exist_ok=True)

rows = []
ds = load_dataset(DATASET_ID, split="train", streaming=True)

for idx, sample in tqdm(enumerate(ds), total=len(selected_indices)):
    if idx not in selected_indices:
        continue
    image, prompt = extract_image_and_prompt(sample)
    res = audit_with_probs(image, prompt)
    heatmap = res["heatmap"]
    binary = heatmap_to_binary(heatmap, HEATMAP_THRESHOLD, image.size)
    safe_mask = ensure_safe_mask(image, prompt, binary)
    masked_image = apply_mask(image, safe_mask)

    image_path = os.path.join(image_dir, f"{idx}.png")
    mask_path = os.path.join(mask_dir, f"{idx}.png")
    masked_path = os.path.join(masked_dir, f"{idx}.png")

    image.save(image_path)
    Image.fromarray(safe_mask).save(mask_path)
    masked_image.save(masked_path)

    rows.append({
        "index": idx,
        "prompt": prompt,
        "category": res["prediction"],
        "category_confidence": float(res["prediction_confidence"]),
        "binary_label": 0 if res["prediction"] == "Safe" else 1,
        "original_image": image_path,
        "mask": mask_path,
        "masked_image": masked_path,
    })
    if len(rows) >= 30_000:
        break

print("Prepared rows:", len(rows))

## 6) Upload to Hugging Face
Provide your token when prompted.

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Enter HF token: " )
login(token=HF_TOKEN)

features = Features({
    "index": Value("int64"),
    "prompt": Value("string"),
    "category": ClassLabel(names=CLASS_NAMES),
    "category_confidence": Value("float32"),
    "category_one_hot": Sequence(Value("int8"), length=len(CLASS_NAMES)),
    "original_image": HFImage(),
    "mask": HFImage(),
    "masked_image": HFImage(),
})

dataset = Dataset.from_list(rows, features=features)
dataset.push_to_hub(REPO_ID, token=HF_TOKEN)

print("Upload complete")